In [7]:
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("LLM ready:", llm.model_name)
print("Embeddings ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LLM ready: llama-3.3-70b-versatile
Embeddings ready


In [8]:
raw_text = """
NovaTech was founded in 2015 by Rahul Mehta in Bangalore, India.
The company started with just 5 employees working on AI solutions.
In 2017, NovaTech launched HelpDesk Pro, their flagship AI-powered customer support platform.
HelpDesk Pro uses natural language processing to automatically resolve 70 percent of customer queries without human intervention.
By 2019, NovaTech had grown to 100 employees and expanded to Mumbai.
In 2021, HelpDesk Pro integrated with Slack, Zendesk, Salesforce, and WhatsApp Business.
In 2023, NovaTech raised 15 million dollars in Series B funding led by Sequoia Capital India.
NovaTech won the Best AI Startup award at TechSparks 2023 and was featured in Forbes India 30 Under 30.
Today NovaTech has 250 employees and serves 500 plus enterprise clients.
NovaTech offers three pricing plans: Starter at 29 dollars per month, Growth at 99 dollars per month, and Enterprise with custom pricing.
NovaTech's engineering team works with Python, FastAPI, React, PostgreSQL, and AWS.
Rahul Mehta previously worked at Google and Flipkart before founding NovaTech.
"""

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    length_function=len
)

chunks = text_splitter.split_text(raw_text)

print(f"Total chunks created: {len(chunks)}")
print(f"\nChunk 0: {chunks[0]}")
print(f"\nChunk 1: {chunks[1]}")
print(f"\nChunk overlap example:")
print(f"End of chunk 0: ...{chunks[0][-50:]}")
print(f"Start of chunk 1: {chunks[1][:50]}...")

Total chunks created: 7

Chunk 0: NovaTech was founded in 2015 by Rahul Mehta in Bangalore, India.
The company started with just 5 employees working on AI solutions.

Chunk 1: In 2017, NovaTech launched HelpDesk Pro, their flagship AI-powered customer support platform.

Chunk overlap example:
End of chunk 0: ...ted with just 5 employees working on AI solutions.
Start of chunk 1: In 2017, NovaTech launched HelpDesk Pro, their fla...


In [9]:
vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    collection_name="novatech_langchain"
)

print(f"Vectorstore created")
print(f"Total documents stored: {vectorstore._collection.count()}")

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print(f"Retriever ready - will fetch top 2 chunks per query")

Vectorstore created
Total documents stored: 7
Retriever ready - will fetch top 2 chunks per query


In [10]:
prompt_template = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have that information."
Do not use any outside knowledge.

Context:
{context}

Question: {question}

Answer:""")

def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully")
print("\nChain structure:")
print("question -> retriever -> format_docs -> prompt -> llm -> output")

RAG chain built successfully

Chain structure:
question -> retriever -> format_docs -> prompt -> llm -> output


In [11]:
# Test 1 — question in knowledge base
answer1 = rag_chain.invoke("Who founded NovaTech and when?")
print("Q: Who founded NovaTech and when?")
print(f"A: {answer1}")
print()

# Test 2 — question not in knowledge base
answer2 = rag_chain.invoke("What is NovaTech's annual revenue?")
print("Q: What is NovaTech's annual revenue?")
print(f"A: {answer2}")
print()

# Test 3 — technical question
answer3 = rag_chain.invoke("What technology stack does NovaTech use?")
print("Q: What technology stack does NovaTech use?")
print(f"A: {answer3}")

Q: Who founded NovaTech and when?
A: NovaTech was founded by Rahul Mehta in 2015.

Q: What is NovaTech's annual revenue?
A: I don't have that information.

Q: What technology stack does NovaTech use?
A: I don't have that information.


In [13]:
question = "What technology stack does NovaTech use?"

retrieved_docs = retriever.invoke(question)

print(f"Question: {question}")
print(f"\nRetrieved {len(retrieved_docs)} chunks:")
for i, doc in enumerate(retrieved_docs):
    print(f"\nChunk {i}: {doc.page_content}")

Question: What technology stack does NovaTech use?

Retrieved 2 chunks:

Chunk 0: NovaTech won the Best AI Startup award at TechSparks 2023 and was featured in Forbes India 30 Under 30.
Today NovaTech has 250 employees and serves 500 plus enterprise clients.

Chunk 1: NovaTech was founded in 2015 by Rahul Mehta in Bangalore, India.
The company started with just 5 employees working on AI solutions.


In [14]:
# Fix — increase k to retrieve more chunks
retriever_k3 = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# Rebuild chain with new retriever
rag_chain_k3 = (
    {"context": retriever_k3 | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

# Check what k=3 retrieves now
retrieved_docs_k3 = retriever_k3.invoke("What technology stack does NovaTech use?")
print("Retrieved chunks with k=3:")
for i, doc in enumerate(retrieved_docs_k3):
    print(f"\nChunk {i}: {doc.page_content}")

print("\n" + "="*50)
answer = rag_chain_k3.invoke("What technology stack does NovaTech use?")
print(f"\nAnswer with k=3: {answer}")

Retrieved chunks with k=3:

Chunk 0: NovaTech won the Best AI Startup award at TechSparks 2023 and was featured in Forbes India 30 Under 30.
Today NovaTech has 250 employees and serves 500 plus enterprise clients.

Chunk 1: NovaTech was founded in 2015 by Rahul Mehta in Bangalore, India.
The company started with just 5 employees working on AI solutions.

Chunk 2: NovaTech's engineering team works with Python, FastAPI, React, PostgreSQL, and AWS.
Rahul Mehta previously worked at Google and Flipkart before founding NovaTech.


Answer with k=3: NovaTech's engineering team works with Python, FastAPI, React, PostgreSQL, and AWS.


RAG debugging observation:

Problem: k=2 missed the tech stack chunk entirely
Fix: increasing k=3 retrieved the right chunk

Real engineering tradeoff:
- k=2 → fast, cheap, but misses relevant chunks
- k=3 → found the answer, slightly more tokens
- k=5+ → more coverage but LLM gets confused with too much context
- Production sweet spot: k=3 to k=5

Key insight: RAG failures are usually retrieval failures,
not generation failures. Always debug retrieval first
before blaming the LLM.



In [17]:
from langchain_community.document_loaders import PyPDFLoader

# In real usage it would be:
# loader = PyPDFLoader("your_file.pdf")
# pages = loader.load()

pages = [
    Document(page_content="Chapter 1: Introduction to AI. Artificial Intelligence is the simulation of human intelligence by machines. It includes learning, reasoning and self-correction.", metadata={"page": 1}),
    Document(page_content="Chapter 2: Machine Learning. Machine learning is a subset of AI that enables systems to learn from data without being explicitly programmed. Key types are supervised, unsupervised and reinforcement learning.", metadata={"page": 2}),
    Document(page_content="Chapter 3: Deep Learning. Deep learning uses neural networks with many layers to learn representations of data. It powers image recognition, speech recognition and natural language processing.", metadata={"page": 3}),
    Document(page_content="Chapter 4: RAG Systems. Retrieval Augmented Generation combines information retrieval with text generation. It grounds LLM responses in factual documents reducing hallucination significantly.", metadata={"page": 4}),
    Document(page_content="Chapter 5: Future of AI. AI is expected to transform industries including healthcare, education, finance and transportation. Responsible AI development requires fairness, transparency and accountability.", metadata={"page": 5}),
]

# Split pages into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)

pdf_chunks = text_splitter.split_documents(pages)

print(f"PDF pages: {len(pages)}")
print(f"Chunks after splitting: {len(pdf_chunks)}")
print(f"\nSample chunk:")
print(f"Content: {pdf_chunks[0].page_content}")
print(f"Metadata: {pdf_chunks[0].metadata}")

PDF pages: 5
Chunks after splitting: 7

Sample chunk:
Content: Chapter 1: Introduction to AI. Artificial Intelligence is the simulation of human intelligence by machines. It includes learning, reasoning and self-correction.
Metadata: {'page': 1}


In [18]:
# Store PDF chunks in a new vectorstore
pdf_vectorstore = Chroma.from_documents(
    documents=pdf_chunks,
    embedding=embeddings,
    collection_name="ai_textbook"
)

# Create retriever
pdf_retriever = pdf_vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# Build RAG chain
pdf_rag_chain = (
    {"context": pdf_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

# Test with questions
questions = [
    "What is machine learning?",
    "How does RAG reduce hallucination?",
    "What industries will AI transform?"
]

for q in questions:
    answer = pdf_rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

Q: What is machine learning?
A: Machine learning is a subset of AI that enables systems to learn from data without being explicitly programmed.

Q: How does RAG reduce hallucination?
A: RAG reduces hallucination significantly by grounding LLM responses in factual documents.

Q: What industries will AI transform?
A: AI is expected to transform industries including healthcare, education, finance, and transportation.



LangChain abstractions vs manual code:
- Chroma.from_texts() → replaces encode + create_collection + add
- as_retriever() → replaces collection.query()
- ChatPromptTemplate → replaces manual f-string prompt building
- StrOutputParser() → replaces response.choices[0].message.content
- Pipe operator | → connects all components cleanly

Key debugging lesson:
- RAG failures = retrieval failures first, not generation
- Always check what chunks are retrieved before blaming LLM
- Fix: increase k, improve chunking, better embeddings

PDF RAG:
- Document object carries metadata (page number, source)
- Metadata preserved through chunking automatically
- split_documents() for Document objects
- split_text() for plain strings

Now I understand what LangChain does under the hood
because I built the same thing from scratch on Day 3.